In [1]:
from pathlib import Path
import sys

import polars as pl
from IPython.display import display


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Unable to locate the repository root from the notebook working directory.")


REPO_ROOT = _find_repo_root(Path.cwd())
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

CACHE_ROOT = REPO_ROOT / "cache"

from wind_datasets.models import TaskSpec
from wind_datasets.persistence import PersistenceExperimentResult, run_persistence_experiment

# Persistence baselines operate on turbine-level series even though the default task is farm-level.
PERSISTENCE_TASK = TaskSpec.next_6h_from_24h(granularity="turbine")


def _round_metrics(frame: pl.DataFrame) -> pl.DataFrame:
    rounded_columns = [column for column in ["mae_kw", "rmse_kw", "mae_pu", "rmse_pu"] if column in frame.columns]
    if not rounded_columns:
        return frame
    return frame.with_columns([pl.col(column).round(3) for column in rounded_columns])


def display_persistence_result(result: PersistenceExperimentResult) -> PersistenceExperimentResult:
    summary = _round_metrics(result.summary).with_columns(
        [
            pl.col("start_timestamp").dt.strftime("%Y-%m-%d %H:%M:%S"),
            pl.col("end_timestamp").dt.strftime("%Y-%m-%d %H:%M:%S"),
        ]
    )
    per_horizon = _round_metrics(result.per_horizon.sort("horizon_step"))
    per_turbine = _round_metrics(result.per_turbine)
    display(summary)
    display(per_horizon)
    display(per_turbine)
    return result


In [2]:
# kelmarsh
kelmarsh_result = run_persistence_experiment(
    "kelmarsh",
    cache_root=CACHE_ROOT,
    task_spec=PERSISTENCE_TASK,
)
display_persistence_result(kelmarsh_result)


dataset_id,quality_profile,rated_power_kw,turbines,eligible_windows,prediction_count,mae_kw,rmse_kw,mae_pu,rmse_pu,start_timestamp,end_timestamp
str,str,f64,i64,i64,i64,f64,f64,f64,f64,str,str
"""kelmarsh""","""default""",2050.0,6,2641667,95100012,247.394,384.74,0.121,0.188,"""2016-01-03 00:00:00""","""2024-12-31 23:50:00"""


horizon_step,horizon_minutes,n_predictions,mae_kw,rmse_kw,mae_pu,rmse_pu
i64,i64,i64,f64,f64,f64,f64
1,10,2641667,93.0,153.675,0.045,0.075
2,20,2641667,124.905,203.042,0.061,0.099
3,30,2641667,143.617,230.915,0.07,0.113
4,40,2641667,157.468,251.115,0.077,0.122
5,50,2641667,168.787,267.445,0.082,0.13
…,…,…,…,…,…,…
32,320,2641667,316.293,468.623,0.154,0.229
33,330,2641667,320.027,473.421,0.156,0.231
34,340,2641667,323.618,478.043,0.158,0.233


turbine_id,rated_power_kw,eligible_windows,prediction_count,mae_kw,rmse_kw,mae_pu,rmse_pu
str,f64,i64,i64,f64,f64,f64,f64
"""Kelmarsh 1""",2050.0,442592,15933312,250.655,387.038,0.122,0.189
"""Kelmarsh 2""",2050.0,444783,16012188,258.902,394.363,0.126,0.192
"""Kelmarsh 3""",2050.0,441356,15888816,256.359,396.447,0.125,0.193
"""Kelmarsh 4""",2050.0,438563,15788268,244.331,379.266,0.119,0.185
"""Kelmarsh 5""",2050.0,437738,15758568,249.71,390.871,0.122,0.191
"""Kelmarsh 6""",2050.0,436635,15718860,224.059,358.857,0.109,0.175


PersistenceExperimentResult(summary=shape: (1, 12)
┌────────────┬────────────┬────────────┬──────────┬───┬─────────┬──────────┬───────────┬───────────┐
│ dataset_id ┆ quality_pr ┆ rated_powe ┆ turbines ┆ … ┆ mae_pu  ┆ rmse_pu  ┆ start_tim ┆ end_times │
│ ---        ┆ ofile      ┆ r_kw       ┆ ---      ┆   ┆ ---     ┆ ---      ┆ estamp    ┆ tamp      │
│ str        ┆ ---        ┆ ---        ┆ i64      ┆   ┆ f64     ┆ f64      ┆ ---       ┆ ---       │
│            ┆ str        ┆ f64        ┆          ┆   ┆         ┆          ┆ datetime[ ┆ datetime[ │
│            ┆            ┆            ┆          ┆   ┆         ┆          ┆ μs]       ┆ μs]       │
╞════════════╪════════════╪════════════╪══════════╪═══╪═════════╪══════════╪═══════════╪═══════════╡
│ kelmarsh   ┆ default    ┆ 2050.0     ┆ 6        ┆ … ┆ 0.12068 ┆ 0.187678 ┆ 2016-01-0 ┆ 2024-12-3 │
│            ┆            ┆            ┆          ┆   ┆         ┆          ┆ 3         ┆ 1         │
│            ┆            ┆            ┆

In [3]:
# penmanshiel
penmanshiel_result = run_persistence_experiment(
    "penmanshiel",
    cache_root=CACHE_ROOT,
    task_spec=PERSISTENCE_TASK,
)
display_persistence_result(penmanshiel_result)


dataset_id,quality_profile,rated_power_kw,turbines,eligible_windows,prediction_count,mae_kw,rmse_kw,mae_pu,rmse_pu,start_timestamp,end_timestamp
str,str,f64,i64,i64,i64,f64,f64,f64,f64,str,str
"""penmanshiel""","""default""",2050.0,14,5405463,194596668,280.148,452.334,0.137,0.221,"""2016-06-02 17:40:00""","""2024-12-31 23:50:00"""


horizon_step,horizon_minutes,n_predictions,mae_kw,rmse_kw,mae_pu,rmse_pu
i64,i64,i64,f64,f64,f64,f64
1,10,5405463,93.841,163.922,0.046,0.08
2,20,5405463,129.778,224.378,0.063,0.109
3,30,5405463,151.008,257.312,0.074,0.126
4,40,5405463,166.915,281.487,0.081,0.137
5,50,5405463,180.345,301.687,0.088,0.147
…,…,…,…,…,…,…
32,320,5405463,368.081,558.016,0.18,0.272
33,330,5405463,372.857,564.051,0.182,0.275
34,340,5405463,377.536,570.0,0.184,0.278


turbine_id,rated_power_kw,eligible_windows,prediction_count,mae_kw,rmse_kw,mae_pu,rmse_pu
str,f64,i64,i64,f64,f64,f64,f64
"""Penmanshiel 01""",2050.0,373157,13433652,284.58,455.726,0.139,0.222
"""Penmanshiel 02""",2050.0,372824,13421664,282.995,455.297,0.138,0.222
"""Penmanshiel 04""",2050.0,370240,13328640,280.819,452.658,0.137,0.221
"""Penmanshiel 05""",2050.0,372353,13404708,278.401,450.076,0.136,0.22
"""Penmanshiel 06""",2050.0,371045,13357620,275.941,451.497,0.135,0.22
…,…,…,…,…,…,…,…
"""Penmanshiel 11""",2050.0,412236,14840496,277.775,445.994,0.135,0.218
"""Penmanshiel 12""",2050.0,417785,15040260,294.18,469.169,0.144,0.229
"""Penmanshiel 13""",2050.0,418476,15065136,297.512,474.182,0.145,0.231


PersistenceExperimentResult(summary=shape: (1, 12)
┌────────────┬────────────┬───────────┬──────────┬───┬──────────┬──────────┬───────────┬───────────┐
│ dataset_id ┆ quality_pr ┆ rated_pow ┆ turbines ┆ … ┆ mae_pu   ┆ rmse_pu  ┆ start_tim ┆ end_times │
│ ---        ┆ ofile      ┆ er_kw     ┆ ---      ┆   ┆ ---      ┆ ---      ┆ estamp    ┆ tamp      │
│ str        ┆ ---        ┆ ---       ┆ i64      ┆   ┆ f64      ┆ f64      ┆ ---       ┆ ---       │
│            ┆ str        ┆ f64       ┆          ┆   ┆          ┆          ┆ datetime[ ┆ datetime[ │
│            ┆            ┆           ┆          ┆   ┆          ┆          ┆ μs]       ┆ μs]       │
╞════════════╪════════════╪═══════════╪══════════╪═══╪══════════╪══════════╪═══════════╪═══════════╡
│ penmanshie ┆ default    ┆ 2050.0    ┆ 14       ┆ … ┆ 0.136658 ┆ 0.220651 ┆ 2016-06-0 ┆ 2024-12-3 │
│ l          ┆            ┆           ┆          ┆   ┆          ┆          ┆ 2         ┆ 1         │
│            ┆            ┆           ┆ 

In [4]:
# hill_of_towie
hill_of_towie_result = run_persistence_experiment(
    "hill_of_towie",
    cache_root=CACHE_ROOT,
    task_spec=PERSISTENCE_TASK,
)
display_persistence_result(hill_of_towie_result)


dataset_id,quality_profile,rated_power_kw,turbines,eligible_windows,prediction_count,mae_kw,rmse_kw,mae_pu,rmse_pu,start_timestamp,end_timestamp
str,str,f64,i64,i64,i64,f64,f64,f64,f64,str,str
"""hill_of_towie""","""default""",2300.0,21,9115020,328140720,297.995,490.502,0.13,0.213,"""2016-01-01 00:00:00""","""2024-09-01 00:00:00"""


horizon_step,horizon_minutes,n_predictions,mae_kw,rmse_kw,mae_pu,rmse_pu
i64,i64,i64,f64,f64,f64,f64
1,10,9115020,109.176,192.066,0.047,0.084
2,20,9115020,151.708,264.36,0.066,0.115
3,30,9115020,176.017,304.142,0.077,0.132
4,40,9115020,193.276,331.56,0.084,0.144
5,50,9115020,207.027,352.745,0.09,0.153
…,…,…,…,…,…,…
32,320,9115020,378.302,589.561,0.164,0.256
33,330,9115020,382.626,595.128,0.166,0.259
34,340,9115020,386.862,600.604,0.168,0.261


turbine_id,rated_power_kw,eligible_windows,prediction_count,mae_kw,rmse_kw,mae_pu,rmse_pu
str,f64,i64,i64,f64,f64,f64,f64
"""T01""",2300.0,435469,15676884,306.241,498.027,0.133,0.217
"""T02""",2300.0,435035,15661260,293.347,484.486,0.128,0.211
"""T03""",2300.0,435446,15676056,311.699,503.851,0.136,0.219
"""T04""",2300.0,434831,15653916,315.918,510.4,0.137,0.222
"""T05""",2300.0,435032,15661152,302.802,493.616,0.132,0.215
…,…,…,…,…,…,…,…
"""T17""",2300.0,432638,15574968,269.246,454.235,0.117,0.197
"""T18""",2300.0,433801,15616836,268.135,454.499,0.117,0.198
"""T19""",2300.0,434607,15645852,313.027,505.986,0.136,0.22


PersistenceExperimentResult(summary=shape: (1, 12)
┌────────────┬────────────┬───────────┬──────────┬───┬──────────┬──────────┬───────────┬───────────┐
│ dataset_id ┆ quality_pr ┆ rated_pow ┆ turbines ┆ … ┆ mae_pu   ┆ rmse_pu  ┆ start_tim ┆ end_times │
│ ---        ┆ ofile      ┆ er_kw     ┆ ---      ┆   ┆ ---      ┆ ---      ┆ estamp    ┆ tamp      │
│ str        ┆ ---        ┆ ---       ┆ i64      ┆   ┆ f64      ┆ f64      ┆ ---       ┆ ---       │
│            ┆ str        ┆ f64       ┆          ┆   ┆          ┆          ┆ datetime[ ┆ datetime[ │
│            ┆            ┆           ┆          ┆   ┆          ┆          ┆ μs]       ┆ μs]       │
╞════════════╪════════════╪═══════════╪══════════╪═══╪══════════╪══════════╪═══════════╪═══════════╡
│ hill_of_to ┆ default    ┆ 2300.0    ┆ 21       ┆ … ┆ 0.129563 ┆ 0.213262 ┆ 2016-01-0 ┆ 2024-09-0 │
│ wie        ┆            ┆           ┆          ┆   ┆          ┆          ┆ 1         ┆ 1         │
│            ┆            ┆           ┆ 

In [5]:
# sdwpf_kddcup
sdwpf_kddcup_result = run_persistence_experiment(
    "sdwpf_kddcup",
    cache_root=CACHE_ROOT,
    quality_profile="default",
    task_spec=PERSISTENCE_TASK,
)
display_persistence_result(sdwpf_kddcup_result)


dataset_id,quality_profile,rated_power_kw,turbines,eligible_windows,prediction_count,mae_kw,rmse_kw,mae_pu,rmse_pu,start_timestamp,end_timestamp
str,str,f64,i64,i64,i64,f64,f64,f64,f64,str,str
"""sdwpf_kddcup""","""default""",1500.0,134,269772,9711792,243.908,352.409,0.163,0.235,"""2020-05-01 00:00:00""","""2020-12-31 23:50:00"""


horizon_step,horizon_minutes,n_predictions,mae_kw,rmse_kw,mae_pu,rmse_pu
i64,i64,i64,f64,f64,f64,f64
1,10,269772,75.22,120.455,0.05,0.08
2,20,269772,104.769,163.493,0.07,0.109
3,30,269772,125.527,191.861,0.084,0.128
4,40,269772,142.639,214.276,0.095,0.143
5,50,269772,157.267,233.107,0.105,0.155
…,…,…,…,…,…,…
32,320,269772,320.654,438.862,0.214,0.293
33,330,269772,325.213,444.783,0.217,0.297
34,340,269772,329.993,450.974,0.22,0.301


turbine_id,rated_power_kw,eligible_windows,prediction_count,mae_kw,rmse_kw,mae_pu,rmse_pu
str,f64,i64,i64,f64,f64,f64,f64
"""1""",1500.0,3655,131580,232.359,365.875,0.155,0.244
"""2""",1500.0,2012,72432,264.757,384.474,0.177,0.256
"""3""",1500.0,2446,88056,255.181,365.283,0.17,0.244
"""4""",1500.0,1978,71208,275.067,393.07,0.183,0.262
"""5""",1500.0,2466,88776,257.909,379.324,0.172,0.253
…,…,…,…,…,…,…,…
"""130""",1500.0,2248,80928,266.212,381.025,0.177,0.254
"""131""",1500.0,2623,94428,244.276,358.498,0.163,0.239
"""132""",1500.0,1779,64044,212.992,299.03,0.142,0.199


PersistenceExperimentResult(summary=shape: (1, 12)
┌────────────┬────────────┬────────────┬──────────┬───┬──────────┬─────────┬───────────┬───────────┐
│ dataset_id ┆ quality_pr ┆ rated_powe ┆ turbines ┆ … ┆ mae_pu   ┆ rmse_pu ┆ start_tim ┆ end_times │
│ ---        ┆ ofile      ┆ r_kw       ┆ ---      ┆   ┆ ---      ┆ ---     ┆ estamp    ┆ tamp      │
│ str        ┆ ---        ┆ ---        ┆ i64      ┆   ┆ f64      ┆ f64     ┆ ---       ┆ ---       │
│            ┆ str        ┆ f64        ┆          ┆   ┆          ┆         ┆ datetime[ ┆ datetime[ │
│            ┆            ┆            ┆          ┆   ┆          ┆         ┆ μs]       ┆ μs]       │
╞════════════╪════════════╪════════════╪══════════╪═══╪══════════╪═════════╪═══════════╪═══════════╡
│ sdwpf_kddc ┆ default    ┆ 1500.0     ┆ 134      ┆ … ┆ 0.162605 ┆ 0.23494 ┆ 2020-05-0 ┆ 2020-12-3 │
│ up         ┆            ┆            ┆          ┆   ┆          ┆         ┆ 1         ┆ 1         │
│            ┆            ┆            ┆